In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightning --quiet

In [ ]:
"""N-HiTS v2 - EDA 정제 피처 + Class Weight + 다변량 직접 처리"""
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor

warnings.filterwarnings("ignore")
pl.seed_everything(42)
print(f"PyTorch: {torch.__version__}, Lightning: {pl.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "nhits_v2"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

TRAIN_END, VAL_START, VAL_END, TEST_START = "2024-06-30", "2024-07-01", "2024-12-31", "2025-01-01"
MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 256
MAX_EPOCHS = 50
LEARNING_RATE = 5e-4
HIDDEN_SIZE = 256
N_STACKS = 3
N_LAYERS_PER_BLOCK = 2
POOLING_SIZES = [2, 3, 5]
DROPOUT = 0.2
GRADIENT_CLIP_VAL = 0.1

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]
print(f"N-HiTS v2: {len(CLEANED_FEATURES)} features, hidden={HIDDEN_SIZE}, stacks={N_STACKS}")

In [ ]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
N_FEATURES = len(CLEANED_FEATURES)

n0 = (df["target_5d"]==0).sum(); n1 = (df["target_5d"]==1).sum()
CLASS_WEIGHTS = torch.tensor([len(df)/(2*n0), len(df)/(2*n1)], dtype=torch.float32)
print(f"rows: {len(df):,} | features: {N_FEATURES} | weights: {CLASS_WEIGHTS.tolist()}")

In [ ]:
# ===== 분할 =====
train_df = df[df["date"]<=TRAIN_END]
print(f"학습: {len(train_df):,} | 검증: {len(df[(df['date']>=VAL_START)&(df['date']<=VAL_END)]):,} | 테스트: {len(df[df['date']>=TEST_START]):,}")

In [ ]:
# ===== N-HiTS v2 모델 =====
class NHiTSBlock(nn.Module):
    def __init__(self, in_size, hidden, out_size, pool_size, n_layers, drop):
        super().__init__()
        pooled = in_size // pool_size
        self.pool = nn.AdaptiveAvgPool1d(pooled)
        layers = [nn.Linear(pooled, hidden), nn.GELU(), nn.Dropout(drop)]
        for _ in range(n_layers-1):
            layers += [nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(drop)]
        layers.append(nn.Linear(hidden, out_size))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(self.pool(x.unsqueeze(1)).squeeze(1))

class NHiTSv2(pl.LightningModule):
    def __init__(self, n_feat, seq_len=30, hidden=256, n_stacks=3, n_layers=2,
                 pools=None, drop=0.2, n_cls=2, lr=5e-4, cw=None):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"])
        self.lr = lr
        in_size = seq_len * n_feat  # flatten
        bo = hidden // n_stacks
        pools = pools or [2,3,5]
        self.stacks = nn.ModuleList([NHiTSBlock(in_size, hidden, bo, pools[i%len(pools)], n_layers, drop) for i in range(n_stacks)])
        self.head = nn.Sequential(nn.Linear(bo*n_stacks, hidden), nn.GELU(), nn.Dropout(drop), nn.Linear(hidden, n_cls))
        self.loss_fn = nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()

    def forward(self, x):
        B = x.shape[0]
        x = x.reshape(B, -1)
        return self.head(torch.cat([s(x) for s in self.stacks], dim=-1))

    def training_step(self, batch, _):
        x,y = batch; loss=self.loss_fn(self(x),y); self.log("train_loss",loss,prog_bar=True); return loss
    def validation_step(self, batch, _):
        x,y = batch; logits=self(x); self.log("val_loss",self.loss_fn(logits,y),prog_bar=True)
        self.log("val_acc",(torch.argmax(logits,1)==y).float().mean(),prog_bar=True)
    def configure_optimizers(self):
        opt=torch.optim.AdamW(self.parameters(),lr=self.lr,weight_decay=1e-4)
        sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,"min",0.5,patience=5,min_lr=1e-6)
        return {"optimizer":opt,"lr_scheduler":{"scheduler":sch,"monitor":"val_loss"}}

print("N-HiTS v2 정의 완료")

In [ ]:
# ===== 데이터셋 =====
def make_samples(full_df, start, end, seq_len, feats):
    samples=[]; s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e: samples.append((v[i-seq_len:i],t[i]))
    return samples

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

tr=SeqDS(make_samples(df,"2008-01-01",TRAIN_END,MAX_ENCODER_LENGTH,CLEANED_FEATURES))
va=SeqDS(make_samples(df,VAL_START,VAL_END,MAX_ENCODER_LENGTH,CLEANED_FEATURES))
te=SeqDS(make_samples(df,TEST_START,None,MAX_ENCODER_LENGTH,CLEANED_FEATURES))
trl=DataLoader(tr,batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
val=DataLoader(va,batch_size=BATCH_SIZE*2,shuffle=False,num_workers=0)
tel=DataLoader(te,batch_size=BATCH_SIZE*2,shuffle=False,num_workers=0)
print(f"학습: {len(tr):,} | 검증: {len(va):,} | 테스트: {len(te):,}")

In [ ]:
# ===== 모델 초기화 =====
model = NHiTSv2(N_FEATURES, MAX_ENCODER_LENGTH, HIDDEN_SIZE, N_STACKS,
                N_LAYERS_PER_BLOCK, POOLING_SIZES, DROPOUT, 2, LEARNING_RATE, CLASS_WEIGHTS)
print(f"파라미터: {sum(p.numel() for p in model.parameters())/1e3:.1f}K")

In [ ]:
# ===== 학습 =====
es = EarlyStopping(monitor="val_loss", patience=8, verbose=True, mode="min")
trainer = pl.Trainer(max_epochs=MAX_EPOCHS, accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1, gradient_clip_val=GRADIENT_CLIP_VAL,
    callbacks=[es, LearningRateMonitor("epoch")], enable_progress_bar=True, log_every_n_steps=50)
print("학습 시작...")
trainer.fit(model, train_dataloaders=trl, val_dataloaders=val)
print(f"완료! Best val_loss: {es.best_score:.4f}")

In [ ]:
# ===== 검증 평가 =====
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
best = NHiTSv2.load_from_checkpoint(trainer.checkpoint_callback.best_model_path, strict=False)
best.eval(); best.cuda()

def predict_all(m, loader):
    ps,ls=[],[]
    with torch.no_grad():
        for x,y in loader:
            ps.extend(torch.softmax(m(x.cuda()),dim=-1)[:,1].cpu().numpy()); ls.extend(y.numpy())
    return np.array(ps),np.array(ls)

vp,va2=predict_all(best,val); vb=(vp>=0.5).astype(int)
da=accuracy_score(va2,vb)
try: auc=roc_auc_score(va2,vp)
except: auc=float("nan")
print("="*60)
print(f"  N-HiTS v2 검증: DA={da*100:.2f}%, AUC={auc:.4f}, 양성={vb.mean()*100:.1f}%")
print("="*60)
print(classification_report(va2,vb,target_names=["하락","상승"]))
print(f"비교: LSTM=48.9%, LightGBM=50.8%")

In [ ]:
# ===== 테스트 =====
tp,ta2=predict_all(best,tel); tb=(tp>=0.5).astype(int)
tda=accuracy_score(ta2,tb)
try: tauc=roc_auc_score(ta2,tp)
except: tauc=float("nan")
print("="*60)
print(f"  N-HiTS v2 테스트: DA={tda*100:.2f}%, AUC={tauc:.4f}, 양성={tb.mean()*100:.1f}%")
print("="*60)
print(classification_report(ta2,tb,target_names=["하락","상승"]))

In [ ]:
# ===== 저장 =====
import json; from datetime import datetime
trainer.save_checkpoint(str(MODEL_SAVE_PATH/"best.ckpt"))
pd.DataFrame({"prob":tp,"actual":ta2,"pred":tb}).to_parquet(str(MODEL_SAVE_PATH/f"pred_{datetime.now().strftime('%Y%m%d')}.parquet"))
json.dump({"model":"NHiTS_v2","val_da":round(da*100,2),"test_da":round(tda*100,2),
  "val_auc":round(float(auc),4),"test_auc":round(float(tauc),4),"features":CLEANED_FEATURES},
  open(str(MODEL_SAVE_PATH/"metrics.json"),"w"),indent=2)
print(f"저장: {MODEL_SAVE_PATH}")

## N-HiTS v2 (정제 피처 + Class Weight + 다변량)\n\n| 모델 | 검증 DA | 테스트 DA | 비고 |\n|------|---------|-----------|------|\n| N-HiTS v1 | 54.2% (fake) | 47.5% | 전부 하락 |\n| **N-HiTS v2** | **?%** | **?%** | 정제+CW+다변량 |\n| LSTM | - | 48.9% | 기존 |\n| LightGBM | - | 50.8% | 기존 최고 |